In [16]:
import dspy
lm = dspy.LM('lm_studio/gemma-3-4b-it', api_base='http://localhost:1234/v1', api_key='LM_STUDIO_API_KEY')
dspy.configure(lm=lm)

In [14]:
math = dspy.ChainOfThought("question -> answer: float")
math(question="Two dice are tossed. What is the probability that the sum equals two?")

Prediction(
    reasoning='When two dice are tossed, there are 6 possible outcomes for each die. This means there are a total of 6 * 6 = 36 possible outcomes. We want to find the probability that the sum of the two dice equals two. The only way this can happen is if both dice show a one (1 + 1 = 2). There is only one outcome where the sum is two, which is (1, 1). Therefore, the probability is the number of favorable outcomes divided by the total number of possible outcomes: Probability = 1/36.',
    answer=0.02777777777777778
)

In [21]:
from dspy.datasets import HotPotQA
HotPotQA(train_seed=2024, train_size=500)


c:\Develop\github\test.dspy\.venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dnhb\.cache\huggingface\hub\datasets--hotpot_qa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 7405/7405 [00:01<00:00, 5327.49 examples/s]


In [24]:

import dspy
from dspy.datasets import HotPotQA


def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=3)
    return [x['text'] for x in results]

trainset = [x.with_inputs('question') for x in HotPotQA(train_seed=2024, train_size=500).train]
react = dspy.ReAct("question -> answer", tools=[search_wikipedia])

tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)
optimized_react = tp.compile(react, trainset=trainset)

2025/04/14 11:29:25 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 100

2025/04/14 11:29:28 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/04/14 11:29:28 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/04/14 11:29:28 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 11%|█         | 11/100 [01:29<12:02,  8.12s/it]
2025/04/14 11:30:58 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/04/14 11:30:58 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
2025/04/14 11:30:58 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing instructions...



Bootstrapped 4 full traces after 11 examples for up to 1 rounds, amounting to 11 attempts.


2025/04/14 11:32:10 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2025/04/14 11:32:10 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given the fields `question`, produce the fields `answer`.

You will be given `question` and your goal is to finish with `answer`.

To do this, you will interleave Thought, Tool Name, and Tool Args, and receive a resulting Observation.

Thought can reason about the current situation, and Tool Name can be the following types:

(1) search_wikipedia. It takes arguments {'query': {'type': 'string'}} in JSON format.
(2) finish, whose description is <desc>Signals that the final outputs, i.e. `answer`, are now available and marks the task as complete.</desc>. It takes arguments {'kwargs': 'Any'} in JSON format.

2025/04/14 11:32:10 INFO dspy.teleprompt.mipro_optimizer_v2: 1: You are an expert researcher tasked with answering factual questions. Given a question, you will use the `search_wikipedia` tool to gather information and then

...

Average Metric: 10.00 / 39 (25.6%):  42%|████▏     | 42/100 [00:29<00:35,  1.65it/s]

2025/04/14 12:31:20 ERROR dspy.utils.parallelizer: Error for Example({'question': 'Who was born first, Wilfred Noy or Keanu Reeves?', 'answer': 'Wilfred Noy'}) (input_keys={'question'}): 'list' object has no attribute 'items'. Set `provide_traceback=True` for traceback.


Average Metric: 24.00 / 96 (25.0%): 100%|██████████| 100/100 [01:35<00:00,  1.05it/s]

2025/04/14 12:32:25 INFO dspy.evaluate.evaluate: Average Metric: 24.0 / 100 (24.0%)
2025/04/14 12:32:25 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [22.0, 24.0, 24.0]
2025/04/14 12:32:25 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 24.0
2025/04/14 12:32:25 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/04/14 12:32:25 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/04/14 12:32:25 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 24.0!


2025/04/14 12:32:25 ERROR dspy.utils.parallelizer: Error for Example({'question': 'Which magazine established itself as a cradle of New Journalism, St. Anthony Messenger or New York?', 'answer': 'New York'}) (input_keys={'question'}): 'list' object has no attribute 'items'. Set `provide_traceback=True` for traceback.
